In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, count
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml import Pipeline

## Creating Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("Task2_DataEngineering") \
    .getOrCreate()

## Loading Processed Dataset

In [4]:
df = spark.read.parquet("nifty_10m_fixed.parquet")

## Checking Number of Partitions

In [5]:
print("Number of Partitions:", df.rdd.getNumPartitions())

Number of Partitions: 3


## Dataset Overview

In [6]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

df.printSchema()

Rows : 10000000
Columns : 15
root
 |-- timestamp: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- iv: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- oi: long (nullable = true)
 |-- strike_price: double (nullable = true)
 |-- spot: double (nullable = true)
 |-- datetime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- expiry_type: string (nullable = true)
 |-- strike_type: string (nullable = true)
 |-- option_type: string (nullable = true)



## Missing Values

In [7]:
missing = df.select([
    count(
        when(col(c).isNull() | isnan(c), c)
    ).alias(c)
    if dict(df.dtypes)[c] in ['double','float']
    else count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing.show()

+---------+----+----+---+-----+---+------+---+------------+----+--------+----+-----------+-----------+-----------+
|timestamp|open|high|low|close| iv|volume| oi|strike_price|spot|datetime|date|expiry_type|strike_type|option_type|
+---------+----+----+---+-----+---+------+---+------------+----+--------+----+-----------+-----------+-----------+
|        0|   0|   0|  0|    0|  0|     0|  0|           0|   0|       0|   0|          0|          0|          0|
+---------+----+----+---+-----+---+------+---+------------+----+--------+----+-----------+-----------+-----------+



## Removing Duplicate Records

In [8]:
before = df.count()

df = df.dropDuplicates()

after = df.count()

print("Duplicates Removed :", before-after)

Duplicates Removed : 0


## Feature Engineering

### Numerical Columns

In [10]:
numerical_cols = [
    "open",
    "high",
    "low",
    "close",
    "iv",
    "volume",
    "oi",
    "strike_price",
    "spot"
]

## Categorical Columns

In [11]:
categorical_cols = [
    "expiry_type",
    "strike_type",
    "option_type"
]

## String Indexing

In [12]:
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c+"_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

## One Hot Encoding

In [13]:
encoder = OneHotEncoder(
    inputCols=[c+"_index" for c in categorical_cols],
    outputCols=[c+"_vec" for c in categorical_cols]
)

## Assemble Numerical Features

In [14]:
assembler_numeric = VectorAssembler(
    inputCols=numerical_cols,
    outputCol="numeric_features"
)

## Standard Scaling

In [15]:
scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features",
    withMean=True,
    withStd=True
)

## Final Feature Vector

In [16]:
assembler_final = VectorAssembler(

    inputCols=[

        "scaled_numeric_features",

        "expiry_type_vec",

        "strike_type_vec",

        "option_type_vec"

    ],

    outputCol="features"
)

## Building PySpark Pipeline

In [17]:
pipeline = Pipeline(

    stages=

        indexers +

        [

            encoder,

            assembler_numeric,

            scaler,

            assembler_final

        ]

)

In [18]:
pipeline_model = pipeline.fit(df)
processed_df = pipeline_model.transform(df)
processed_df.select(

    "features"

).show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                                                                                           |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|(25,[0,1,2,3,4,5,6,7,8,9,21,23],[-1.3790010380030868,-1.3832306307090596,-1.375367954049833,-1.3798000336748548,0.7323391680163158,-0.05153725409208124,-0.031632599600142176,-1.630845241473202,-1.7344878844765257,1.0,1.0,1.0]) |
|(25,[0,1,2,3,4,5,6,7,8,9,21,23],[-1.3803138561649966,-1.384536828322025,-1.3763

## Verifying Pipelines Columns

In [19]:
processed_df.printSchema()

root
 |-- timestamp: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- iv: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- oi: long (nullable = true)
 |-- strike_price: double (nullable = true)
 |-- spot: double (nullable = true)
 |-- datetime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- expiry_type: string (nullable = true)
 |-- strike_type: string (nullable = true)
 |-- option_type: string (nullable = true)
 |-- expiry_type_index: double (nullable = false)
 |-- strike_type_index: double (nullable = false)
 |-- option_type_index: double (nullable = false)
 |-- expiry_type_vec: vector (nullable = true)
 |-- strike_type_vec: vector (nullable = true)
 |-- option_type_vec: vector (nullable = true)
 |-- numeric_features: vector (nullable = true)
 |-- scaled_numeric_features: vector (nullable = true)
 |-- features: vector (nullab

In [20]:
processed_df.write \
.mode("overwrite") \
.parquet("task2_preprocessed.parquet")

## Pipeline Stages

In [21]:
print("Pipeline Stages\n")

for stage in pipeline.getStages():
    print(stage)

Pipeline Stages

StringIndexer_2ffb298a2f5e
StringIndexer_399b85318f58
StringIndexer_ee48d82acc77
OneHotEncoder_3e6ce4090a8a
VectorAssembler_7c280a8e423b
StandardScaler_b5a92974f8df
VectorAssembler_873dbfb2d2fb


In [22]:
print("Rows :", processed_df.count())
print("Columns :", len(processed_df.columns))

Rows : 10000000
Columns : 24
